# 03 - Silver to Gold with PySpark SQL

Build dashboard-ready metrics and publish cleaned event data under the processed prefix.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.spark_pipeline import build_gold, create_spark_session, read_parquet_object
from insightsstream.storage import create_store

spark = create_spark_session("InsightsStream-Notebook-Gold")
store = create_store()

In [ ]:
run = build_gold(spark=spark, store=store)
print(asdict(run))

gold = read_parquet_object(spark, store, run.gold_key)
gold.createOrReplaceTempView("gold_daily_metrics")
gold.show(50, truncate=False)

In [ ]:
spark.sql("""
    SELECT
        metric,
        SUM(event_count) AS events,
        ROUND(SUM(total_value), 2) AS total_value,
        SUM(unique_entities) AS entity_metric_memberships
    FROM gold_daily_metrics
    GROUP BY metric
    ORDER BY total_value DESC
""").show(truncate=False)

print(f"Processed output: {run.export_key}")